In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()   # already connected — no password needed inside Snowflake
session.sql("USE DATABASE RETAIL_CAPSTONE").collect()   # <-- add this line

fact_sales   = session.table("DW.FACT_SALES").to_pandas()
dim_customer = session.table("DW.DIM_CUSTOMER").filter('"IS_CURRENT" = TRUE').to_pandas()
dim_product  = session.table("DW.DIM_PRODUCT").to_pandas()
dim_date     = session.table("DW.DIM_DATE").to_pandas()
dim_location = session.table("DW.DIM_LOCATION").to_pandas()

print(fact_sales.shape, dim_customer.shape, dim_product.shape)

In [ ]:
df = (fact_sales
      .merge(dim_date, left_on="DATE_SK", right_on="DATE_SK", how="left")
      .merge(dim_product, left_on="PRODUCT_SK", right_on="PRODUCT_SK", how="left")
      .merge(dim_location, left_on="LOCATION_SK", right_on="LOCATION_SK", how="left")
      .merge(dim_customer, left_on="CUSTOMER_SK", right_on="CUSTOMER_SK", how="left"))
df.head()

In [ ]:
# Monthly revenue trend
monthly_rev = df.groupby(["YEAR","MONTH"])["LINE_AMOUNT"].sum().reset_index()
print(monthly_rev, "\n")

# Category-wise revenue
category_rev = df.groupby("CATEGORY")["LINE_AMOUNT"].sum().sort_values(ascending=False)
print(category_rev, "\n")

# Top 10 cities by revenue
top_cities = df.groupby("CITY_y")["LINE_AMOUNT"].sum().sort_values(ascending=False).head(10)
print(top_cities, "\n")

# Customer segments — simple: split by total spend into Low/Medium/High
cust_spend = df.groupby("CUSTOMER_CODE")["LINE_AMOUNT"].sum()
segments = pd.qcut(cust_spend, q=3, labels=["Low","Medium","High"])
segment_counts = segments.value_counts()
print(segment_counts, "\n")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ------------------------------------------------------------
# Global style — set once, applies to every chart below
# ------------------------------------------------------------
sns.set_theme(style="darkgrid", rc={
    "axes.facecolor": "#0f172a",
    "figure.facecolor": "#0f172a",
    "axes.edgecolor": "#334155",
    "axes.labelcolor": "#e2e8f0",
    "text.color": "#f8fafc",
    "xtick.color": "#cbd5e1",
    "ytick.color": "#cbd5e1",
    "grid.color": "#1e293b",
    "font.family": "sans-serif",
})
PALETTE = sns.color_palette(["#38bdf8", "#a78bfa", "#fb7185", "#34d399", "#fbbf24", "#f472b6", "#818cf8", "#2dd4bf"])

def money_fmt(x, pos):
    return f"₹{x/1e7:,.0f}Cr" if x >= 1e7 else f"₹{x/1e5:,.0f}L" if x >= 1e5 else f"₹{x:,.0f}"

fig, axes = plt.subplots(2, 3, figsize=(22, 11))
fig.suptitle("Global Retail E-Commerce — Analytics Overview", fontsize=20, fontweight="bold", color="#f8fafc", y=1.02)

# --- 1. Line chart: Monthly Revenue Trend ---
ax = axes[0, 0]
sns.lineplot(x=monthly_rev.index, y=monthly_rev["LINE_AMOUNT"], ax=ax, color="#38bdf8", linewidth=2.5, marker="o", markersize=4)
ax.fill_between(monthly_rev.index, monthly_rev["LINE_AMOUNT"], alpha=0.15, color="#38bdf8")
ax.set_title("Monthly Revenue Trend", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Month"); ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(money_fmt))

# --- 2. Bar chart: Revenue by Category ---
ax = axes[0, 1]
sns.barplot(x=category_rev.index, y=category_rev.values, hue=category_rev.index, palette=PALETTE, legend=False, ax=ax)
ax.set_title("Revenue by Category", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel(""); ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(money_fmt))
ax.tick_params(axis='x', rotation=30)

# --- 3. Histogram: Distribution of Line Amounts ---
ax = axes[0, 2]
sns.histplot(df["LINE_AMOUNT"], bins=40, ax=ax, color="#38bdf8", edgecolor="#0f172a")
ax.set_title("Distribution of Order Line Amounts", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Line Amount"); ax.set_ylabel("Count")

# --- 4. Pie/Donut chart: Customer Segments by Spend ---
ax = axes[1, 0]
colors = ["#fb7185", "#fbbf24", "#34d399"]
wedges, texts, autotexts = ax.pie(
    segment_counts.values, labels=segment_counts.index, autopct="%1.1f%%",
    colors=colors, startangle=90, pctdistance=0.8,
    wedgeprops={"width": 0.4, "edgecolor": "#0f172a", "linewidth": 2},
    textprops={"color": "#f8fafc", "fontsize": 11}
)
ax.set_title("Customer Segments by Spend", fontsize=14, fontweight="bold", pad=12)

# --- 5. Horizontal bar: Top 10 Cities by Revenue ---
ax = axes[1, 1]
top_cities_sorted = top_cities.sort_values()
sns.barplot(x=top_cities_sorted.values, y=top_cities_sorted.index, ax=ax, color="#38bdf8")
ax.set_title("Top 10 Cities by Revenue", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Revenue"); ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(money_fmt))

# --- 6th panel: turn off (or use for a summary text card) ---
axes[1, 2].axis("off")
axes[1, 2].text(0.5, 0.6, "Data Pipeline", fontsize=16, fontweight="bold", color="#f8fafc", ha="center", transform=axes[1, 2].transAxes)
axes[1, 2].text(0.5, 0.45, "RAW → STAGING → DW\n(Star Schema + SCD Type 2)", fontsize=12, color="#94a3b8", ha="center", transform=axes[1, 2].transAxes)

plt.tight_layout()
plt.show()